# Stage 2 — LSTM Baseline

Goal: a from-scratch LSTM sentiment classifier in PyTorch, trained on the
coffee competitive-set reviews. This is the baseline the transformer must beat.





2.7 Evaluation

## 2.1 Importing libraries

In [25]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Subset
import torch.optim as optim
import pandas as pd
from pathlib import Path
from collections import Counter

## 2.2 Setup and data loading

In [2]:
ROOT = Path.cwd().parents[0]
train = pd.read_parquet(ROOT / "data/processed/coffee_train.parquet")

print(train.shape)
print(train["label"].value_counts(normalize=True).round(3))

(788195, 13)
label
1    0.865
0    0.135
Name: proportion, dtype: float64


## 2.3 Vocabulary

In [3]:
def build_vocab(texts, min_freq=2):
    # texts: list of review strings from the TRAINING set only
    # min_freq: a word must appear at least this many times to earn its own number
    counter = Counter()
    for text in texts:
        counter.update(text.lower().split())   # lowercase, split on spaces

    # two special markers first: padding and unknown
    word_to_idx = {"<pad>": 0, "<unk>": 1}

    for word, count in counter.items():
        if count >= min_freq:
            word_to_idx[word] = len(word_to_idx)   # next free number

    idx_to_word = {idx: word for word, idx in word_to_idx.items()}
    return word_to_idx, idx_to_word


def text_to_ids(text, word_to_idx):
    ids = []
    for word in text.lower().split():
        if word in word_to_idx:
            ids.append(word_to_idx[word])          # known word -> its number
        else:
            ids.append(word_to_idx["<unk>"])       # unknown word -> 1
    return ids


word_to_idx, idx_to_word = build_vocab(train["full_text"].tolist(), min_freq=20)

print("vocabulary size:", len(word_to_idx))
print("first few words:", list(word_to_idx.items())[:8])

vocabulary size: 21309
first few words: [('<pad>', 0), ('<unk>', 1), ('not', 2), ('actually', 3), ('for', 4), ('use', 5), ('in', 6), ('espresso', 7)]


## 2.4 Text to tensors (Dataset + DataLoader)

In [6]:
class ReviewDataset(Dataset):
    def __init__(self, dataframe, word_to_idx, max_len=128):
        self.texts  = dataframe["full_text"].tolist()   # all review strings
        self.labels = dataframe["label"].tolist()       # matching labels
        self.word_to_idx = word_to_idx
        self.max_len = max_len                          # cap very long reviews

    def __len__(self):
        return len(self.labels)                         # how many reviews total

    def __getitem__(self, i):
        # convert review i to token IDs, then truncate to max_len
        ids = text_to_ids(self.texts[i], self.word_to_idx)[:self.max_len]
        # return the IDs and label, both as tensors
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.labels[i], dtype=torch.long)

In [8]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    # batch is a list of (ids_tensor, label_tensor) pairs
    ids_list   = [item[0] for item in batch]   # pull out all the id tensors
    labels     = torch.stack([item[1] for item in batch])  # stack labels into one tensor

    # record true lengths BEFORE padding (the model needs these later)
    lengths = torch.tensor([len(ids) for ids in ids_list])

    # pad every sequence to the longest one in THIS batch, filling with 0 (<pad>)
    padded = pad_sequence(ids_list, batch_first=True, padding_value=0)

    return padded, lengths, labels

In [ ]:
val = pd.read_parquet(ROOT / "data/processed/coffee_val.parquet")

train_ds = ReviewDataset(train, word_to_idx, max_len=128)
val_ds   = ReviewDataset(val,   word_to_idx, max_len=128)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, collate_fn=collate_fn)

# pull ONE batch and inspect it
padded, lengths, labels = next(iter(train_loader))
print("padded shape:", padded.shape)      # expect (64, something <= 128)
print("lengths shape:", lengths.shape)     # expect (64,)
print("labels shape:", labels.shape)       # expect (64,)
print("first sequence length:", lengths[0].item())
print("padded first row (last 20 values):", padded[0][-20:])

padded shape: torch.Size([64, 128])
lengths shape: torch.Size([64])
labels shape: torch.Size([64])
first sequence length: 15
padded first row (last 20 values): tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])


## 2.5 The model

In [16]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_classes=2):
        super().__init__()

        # layer 1 — embedding: turns word IDs into vectors
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # layer 2 — LSTM: reads the sequence, produces a summary vector
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        # layer 3 — the linear layer: the classifier. HERE it is.
        # takes hidden_dim numbers in, produces num_classes scores out
        self.fc = nn.Linear(hidden_dim, num_classes)
        
    def forward(self, x, lengths):
        # x arrives as (batch, seq_len) — the padded token IDs

        # 1. embedding: each ID becomes a vector
        #    (batch, seq_len) -> (batch, seq_len, embed_dim)
        embedded = self.embedding(x)

        # 2. LSTM reads the sequence
        #    output = hidden state at EVERY word
        #    (hidden, cell) = the FINAL states
        output, (hidden, cell) = self.lstm(embedded)

        # 3. grab the summary vector: the final hidden state
        #    hidden shape is (1, batch, hidden_dim) -> squeeze to (batch, hidden_dim)
        summary = hidden.squeeze(0)

        # 4. linear layer: 128 numbers -> 2 class scores
        #    (batch, hidden_dim) -> (batch, num_classes)
        logits = self.fc(summary)

        return logits

In [17]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, lengths):
        embedded = self.embedding(x)
        output, (hidden, cell) = self.lstm(embedded)
        summary = hidden.squeeze(0)
        logits = self.fc(summary)
        return logits


model = LSTMClassifier(vocab_size=len(word_to_idx))
print(model)

# sanity check: push one batch through and check the output shape
logits = model(padded, lengths)
print("logits shape:", logits.shape)   # expect (64, 2)

LSTMClassifier(
  (embedding): Embedding(21309, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)
logits shape: torch.Size([64, 2])


## 2.6 Training loop

In [24]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0

    for padded, lengths, labels in loader:
        padded = padded.to(device)      # move inputs to GPU
        labels = labels.to(device)      # move labels to GPU
        # lengths stays on CPU — it's only used for bookkeeping, not math

        optimizer.zero_grad()
        logits = model(padded, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)

In [26]:
small = Subset(train_ds, range(2000))
small_loader = DataLoader(small, batch_size=64, shuffle=True, collate_fn=collate_fn)
loss = train_one_epoch(model, small_loader, criterion, optimizer)
print("test loss on 2000 reviews:", round(loss, 4))

test loss on 2000 reviews: 0.6933


In [28]:
num_epochs = 5

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, criterion)

    print(f"epoch {epoch}/{num_epochs}  "
          f"train_loss {train_loss:.4f}  "
          f"val_loss {val_loss:.4f}  "
          f"val_acc {val_acc:.3f}")

epoch 1/5  train_loss 0.2834  val_loss 0.1275  val_acc 0.949
epoch 2/5  train_loss 0.0935  val_loss 0.1165  val_acc 0.955
epoch 3/5  train_loss 0.0743  val_loss 0.1174  val_acc 0.955
epoch 4/5  train_loss 0.0609  val_loss 0.1238  val_acc 0.956
epoch 5/5  train_loss 0.0509  val_loss 0.1373  val_acc 0.959


## notebook checks

In [ ]:
# counter = Counter()
# for text in train["full_text"].tolist():
#     counter.update(text.lower().split())

# for mf in [2, 3, 5, 10, 20]:
#     size = sum(1 for w, c in counter.items() if c >= mf) + 2  # +2 for <pad>, <unk>
#     print(f"min_freq={mf:>2} -> vocab size {size}")

min_freq= 2 -> vocab size 113703
min_freq= 3 -> vocab size 77783
min_freq= 5 -> vocab size 52686
min_freq=10 -> vocab size 33128
min_freq=20 -> vocab size 21309


In [ ]:
# total_tokens = sum(counter.values())   # every word occurrence, counted with repeats

# print(f"total tokens: {total_tokens:,}")
# print(f"total unique types: {len(counter):,}\n")

# for mf in [2, 3, 5, 10, 20, 50]:
#     kept_types = [w for w, c in counter.items() if c >= mf]
#     kept_tokens = sum(counter[w] for w in kept_types)
#     coverage = kept_tokens / total_tokens
#     print(f"min_freq={mf:>2}  vocab {len(kept_types)+2:>7,}  token coverage {coverage:.4f}")

total tokens: 24,409,386
total unique types: 307,161

min_freq= 2  vocab 113,703  token coverage 0.9921
min_freq= 3  vocab  77,783  token coverage 0.9891
min_freq= 5  vocab  52,686  token coverage 0.9857
min_freq=10  vocab  33,128  token coverage 0.9805
min_freq=20  vocab  21,309  token coverage 0.9739
min_freq=50  vocab  12,011  token coverage 0.9621


In [ ]:
# lengths = train["full_text"].str.split().str.len()
# print(lengths.describe().round(1))
# print("95th percentile:", lengths.quantile(0.95))
# print("99th percentile:", lengths.quantile(0.99))

count    788195.0
mean         31.0
std          37.6
min           2.0
25%           9.0
50%          20.0
75%          39.0
max        2711.0
Name: full_text, dtype: float64
95th percentile: 95.0
99th percentile: 178.0


In [ ]:
# import numpy as np

# # how many of each class in training
# counts = train["label"].value_counts().sort_index()   # index 0 = negative, 1 = positive
# print("class counts:", counts.to_dict())

# # weight each class inversely to its frequency: rarer class -> bigger weight
# total = counts.sum()
# weights = total / (2 * counts.values)
# class_weights = torch.tensor(weights, dtype=torch.float)
# print("class weights:", class_weights)

class counts: {0: 106216, 1: 681979}
class weights: tensor([3.7103, 0.5779])


In [ ]:
# device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# print("using:", device)

# model = LSTMClassifier(vocab_size=len(word_to_idx)).to(device)
# class_weights = class_weights.to(device)          # loss weights must match too
# criterion = nn.CrossEntropyLoss(weight=class_weights)
# optimizer = optim.Adam(model.parameters(), lr=1e-3)

using: mps


In [27]:
def evaluate(model, loader, criterion):
    model.eval()                      # evaluation mode (turns off dropout etc.)
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():             # don't compute gradients — we're only measuring
        for padded, lengths, labels in loader:
            padded = padded.to(device)
            labels = labels.to(device)

            logits = model(padded, lengths)
            loss = criterion(logits, labels)
            total_loss += loss.item()

            preds = logits.argmax(dim=1)          # higher score wins
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total